# Covertype - Entrenamiento RandomForestClassifier

Este notebook:
1. Consulta la tabla `data_prepared` en MySQL (train/test por `data_type`)
2. Entrena un modelo RandomForestClassifier
3. Guarda el modelo como PKL en MinIO

In [1]:
import os
import pymysql
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from minio import Minio

# Configuración MySQL
MYSQL_HOST = os.environ.get("MYSQL_HOST", "mysql")
MYSQL_PORT = int(os.environ.get("MYSQL_PORT", 3306))
MYSQL_USER = os.environ.get("MYSQL_USER", "airflow_user")
MYSQL_PASSWORD = os.environ.get("MYSQL_PASSWORD", "airflow_pass")
MYSQL_DATABASE = os.environ.get("MYSQL_DATABASE", "airflow_data")

# Configuración MinIO
MINIO_HOST = os.environ.get("MINIO_ENDPOINT", "http://minio:9000").replace("http://", "").replace("https://", "")
MINIO_ACCESS = os.environ.get("MINIO_ACCESS_KEY", "minioadmin")
MINIO_SECRET = os.environ.get("MINIO_SECRET_KEY", "minioadmin")
MINIO_BUCKET = os.environ.get("MINIO_BUCKET", "models")

## 1. Cargar datos desde MySQL

In [2]:
conn = pymysql.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database=MYSQL_DATABASE,
)

df_train = pd.read_sql("SELECT * FROM data_prepared WHERE data_type = 'train'", conn)
df_test = pd.read_sql("SELECT * FROM data_prepared WHERE data_type = 'test'", conn)
conn.close()

print(f"Train: {len(df_train)} registros")
print(f"Test: {len(df_test)} registros")

/tmp/ipykernel_547/1389171964.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_train = pd.read_sql("SELECT * FROM data_prepared WHERE data_type = 'train'", conn)


Train: 23240 registros
Test: 5810 registros


/tmp/ipykernel_547/1389171964.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_test = pd.read_sql("SELECT * FROM data_prepared WHERE data_type = 'test'", conn)


## 2. Preparar features y target

In [3]:
FEATURE_COLS = [
    "elevation", "aspect", "slope",
    "horizontal_distance_to_hydrology", "vertical_distance_to_hydrology",
    "horizontal_distance_to_roadways", "hillshade_9am", "hillshade_noon",
    "hillshade_3pm", "horizontal_distance_to_fire_points",
    "wilderness_area", "soil_type"
]
TARGET_COL = "cover_type"

# Convertir numéricos
numeric_cols = FEATURE_COLS[:10]
for col in numeric_cols:
    df_train[col] = pd.to_numeric(df_train[col], errors="coerce")
    df_test[col] = pd.to_numeric(df_test[col], errors="coerce")

# Codificar categóricos (fit en train+test para evitar labels desconocidos)
le_wilderness = LabelEncoder()
le_soil = LabelEncoder()
all_w = pd.concat([df_train["wilderness_area"], df_test["wilderness_area"]]).fillna("unknown").astype(str)
all_s = pd.concat([df_train["soil_type"], df_test["soil_type"]]).fillna("unknown").astype(str)
le_wilderness.fit(all_w.unique())
le_soil.fit(all_s.unique())

df_train["wilderness_area"] = le_wilderness.transform(df_train["wilderness_area"].fillna("unknown").astype(str))
df_test["wilderness_area"] = le_wilderness.transform(df_test["wilderness_area"].fillna("unknown").astype(str))
df_train["soil_type"] = le_soil.transform(df_train["soil_type"].fillna("unknown").astype(str))
df_test["soil_type"] = le_soil.transform(df_test["soil_type"].fillna("unknown").astype(str))

# Eliminar filas con NaN
df_train = df_train.dropna(subset=FEATURE_COLS + [TARGET_COL])
df_test = df_test.dropna(subset=FEATURE_COLS + [TARGET_COL])

X_train = df_train[FEATURE_COLS]
y_train = df_train[TARGET_COL].astype(int)
X_test = df_test[FEATURE_COLS]
y_test = df_test[TARGET_COL].astype(int)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

X_train: (23240, 12), y_train: (23240,)
X_test: (5810, 12), y_test: (5810,)


## 3. Entrenar RandomForestClassifier

In [4]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

score_train = model.score(X_train, y_train)
score_test = model.score(X_test, y_test)
print(f"Accuracy train: {score_train:.4f}")
print(f"Accuracy test: {score_test:.4f}")

Accuracy train: 1.0000
Accuracy test: 0.9429


## 4. Guardar PKL y subir a MinIO

In [5]:
MODEL_FILENAME = "covertype_rf.pkl"
MODEL_PATH = f"/tmp/{MODEL_FILENAME}"

# Guardar modelo localmente
joblib.dump(model, MODEL_PATH)
print(f"Modelo guardado en {MODEL_PATH}")

# Subir a MinIO
client = Minio(
    MINIO_HOST,
    access_key=MINIO_ACCESS,
    secret_key=MINIO_SECRET,
    secure=False,
)

client.fput_object(
    MINIO_BUCKET,
    MODEL_FILENAME,
    MODEL_PATH,
)

print(f"Modelo subido a MinIO: {MINIO_BUCKET}/{MODEL_FILENAME}")

Modelo guardado en /tmp/covertype_rf.pkl
Modelo subido a MinIO: models/covertype_rf.pkl
